# Amazon Top-K 추천 실험 B187–B195

1. Kaggle 입력 및 공통 Validation / Inner 분할
2. 원본 JSONL 순서 복원과 B192–B194 Ordering 후보 단독 평가
3. B176 후보 준비, 후보 진단 및 B195 LightGBM 결합
4. B187–B191 시간 기반 후보와 재정렬 Ablation
5. Validation 비교, 제출 검증, 공유 ZIP

Competition Data, 교수님 원본 `train_reviews.jsonl`과 `item_meta.jsonl`, 그리고 가능하면 B176 Output 또는 `TEAM_ALL_TOP100.zip`을 Add Input으로 추가하세요. JSONL이 원본 문자열 ID를 사용하면 공식 ID 매핑을 지정해야 합니다. Ordering 후보 생성은 CPU에서 실행합니다. 기존 원천 후보와 임베딩 재계산이 필요하면 GPU를 사용하세요.

기본 실행 순서는 B192–B194 → B195 → B187–B191입니다. Ordering까지만 실행하려면 `RUN_TEMPORAL = False`로 두고 Save Version → Save & Run All을 실행하세요. B176의 기록은 Validation 0.062098, Kaggle 0.06915입니다. B195 결합 윈도우는 Inner Oracle Recall@20으로 선택합니다. Padding에는 해당 학습 분할의 인기도를 사용합니다. Inner 재정렬 출력은 학습 내 예측이 포함되어 manifest에 표시됩니다.


In [1]:
from pathlib import Path
import os,sys,json,subprocess,importlib.util, time
import numpy as np, pandas as pd
import torch


import shutil, zipfile, hashlib
import psutil
from importlib.metadata import version

FEATURE_MODE = 'real'      # B176 로그와 동일한 실제 임베딩. 오프라인 재구성 시 'proxy'
DOWNLOAD_IMAGES = True
CACHE_INPUT = None         # 필요 시 /kaggle/input/<B176-output> 아래 경로로 지정
REUSE_B176 = True
RESUME_CHECKPOINT = None   # 필요 시 이전 B187_191_checkpoint.zip의 절대 경로
ORDERING_JSONL = None       # 자동 탐색 또는 원본 파일의 절대 경로
ORDERING_USER_MAP = None    # 필요 시 raw_id,user_id 컬럼의 공식 매핑 CSV
ORDERING_ITEM_MAP = None    # 필요 시 raw_id,item_id 컬럼의 공식 매핑 CSV
RUN_TEMPORAL = True         # False면 Ordering 실험까지만 진행하고 결과 ZIP 생성
THREADS = min(4, os.cpu_count() or 1)
SEEDS_RANK = [0, 1, 2]
PRESET = 'MAX'
ALLOW_OPTIONAL_FAILURE = False
os.environ['OMP_NUM_THREADS'] = str(THREADS)
os.environ['DS_FEATURE_MODE'] = FEATURE_MODE
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
ROOT = WORK/'team_ensemble'/'D'
print('CPU threads', THREADS, '| RAM GiB', round(psutil.virtual_memory().total/2**30, 1),
      '| disk free GiB', round(shutil.disk_usage(WORK).free/2**30, 1))
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(gpu.name, '| GPU GiB', round(gpu.total_memory/2**30, 1))
else:
    print('CPU 모드. 캐시 없는 real 임베딩과 원천 재학습에는 GPU 권장.')
if importlib.util.find_spec('lightgbm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
for pkg in ['lightgbm','numpy','pandas','scipy','torch','scikit-learn']:
    print(pkg, version(pkg))


CPU threads 4 | RAM GiB 31.3 | disk free GiB 19.5
Tesla T4 | GPU GiB 14.6
lightgbm 4.6.0
numpy 2.1.3
pandas 2.3.3
scipy 1.16.3
torch 2.11.0+cu130
scikit-learn 1.6.1


In [2]:
# 원본 GitHub ZIP에 있는 D/src/*.py와 C의 teamkit.py 및 새 A 재구현 코드를 노트북 안에 내장
# 추가 GitHub 다운로드/비공개 저장소 인증 없이 실행 가능
import base64,io,zipfile,sys
SOURCE_ZIP_B64 = """..."""
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP_B64))) as z:
    z.extractall(ROOT)
print('소스 설치:',len(list((ROOT/'src').glob('*.py'))),'개 파이썬 파일')
sys.path.insert(0,str(ROOT));import pipeline_helpers as P
ROOT=P.ROOT; SRC=P.SRC; OUT=P.OUT
OUT.mkdir(parents=True,exist_ok=True)

소스 설치: 12 개 파이썬 파일


In [3]:
# Kaggle Input 자동 탐색 + 공통 val/inner split 준비
import shutil,zipfile,json
INPUT=Path('/kaggle/input')
assert INPUT.exists(),'Kaggle Notebook 안에서 실행해야 합니다.'
train_candidates=[p for p in INPUT.rglob('train.csv') if (p.parent/'test.csv').exists() and 'validation' not in str(p).lower()]
if not train_candidates:
    # 학습 파일이 ZIP 내부에만 있는 경우 지원
    for zp in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(zp) as z:
                names={Path(x).name:x for x in z.namelist()}
                if 'train.csv' in names and 'test.csv' in names:
                    tmp=WORK/'raw_competition';tmp.mkdir(exist_ok=True)
                    for name in ['train.csv','test.csv']:
                        (tmp/name).write_bytes(z.read(names[name]))
                    train_candidates=[tmp/'train.csv'];break
        except (OSError,zipfile.BadZipFile):pass
assert train_candidates,'train.csv + test.csv를 찾지 못했습니다. Kaggle Competition Data를 Add Input 해주세요.'
TRAIN=sorted(train_candidates,key=lambda p:('competition' not in str(p).lower(),len(str(p))))[0]
TEST=TRAIN.parent/'test.csv'
(ROOT/'data').mkdir(parents=True,exist_ok=True)
for source,dest in [(TRAIN,ROOT/'data/train.csv'),(TEST,ROOT/'data/test.csv')]:
    if not dest.exists():shutil.copy2(source,dest)

meta_candidates=list(INPUT.rglob('item_meta.jsonl'))
if not meta_candidates:
    for zp in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(zp) as z:
                hit=next((x for x in z.namelist() if x.endswith('item_meta.jsonl')),None)
                if hit:
                    (ROOT/'data/item_meta.jsonl').write_bytes(z.read(hit));break
        except (OSError,zipfile.BadZipFile):pass
elif not (ROOT/'data/item_meta.jsonl').exists():
    shutil.copy2(meta_candidates[0],ROOT/'data/item_meta.jsonl')
HAS_META=(ROOT/'data/item_meta.jsonl').is_file()
print('Original train:',TRAIN,'| item_meta:',HAS_META)

VAL=ROOT/'teamkit/validation';VAL.mkdir(parents=True,exist_ok=True)
existing=[p for p in INPUT.rglob('manifest.json') if (p.parent/'val_train.csv').exists() and (p.parent/'val_truth.csv').exists() and (p.parent/'val_users.csv').exists()]
preferred=[]
for p in existing:
    try:
        if json.loads(p.read_text()).get('split_id')=='f0dfad2f4431a876':preferred.append(p)
    except Exception:pass
if preferred and not (VAL/'manifest.json').exists():
    for n in ['manifest.json','val_train.csv','val_truth.csv','val_users.csv']:
        shutil.copy2(preferred[0].parent/n,VAL/n)

sys.path.insert(0,str(ROOT));import teamkit
if not (VAL/'manifest.json').exists():
    manifest=teamkit.make_split(str(ROOT/'data/train.csv'),None,VAL,seed=42)
else:manifest=teamkit.verify_split(VAL)
print('VAL manifest:',manifest['split_id'],'| truth:',manifest.get('truth_rows'))
if manifest['split_id']!='f0dfad2f4431a876':
    raise ValueError('공통 Validation ID가 다릅니다. 반드시 팀 공통 데이터/원본 train.csv를 사용하세요.')
INNER=ROOT/'data/inner';INNER.mkdir(parents=True,exist_ok=True)
if not (INNER/'manifest.json').exists():
    inner_manifest=teamkit.make_split(str(VAL/'val_train.csv'),None,INNER,seed=7)
else:inner_manifest=teamkit.verify_split(INNER)
print('INNER manifest:',inner_manifest['split_id'],'| truth:',inner_manifest.get('truth_rows'))
assert inner_manifest['split_id']=='b45fa59b9ee68d8a','D파트와 동일한 내부 split이 아닙니다.'
for m in ['val','final','inner']:
    path=VAL/'val_users.csv' if m=='val' else ROOT/'data/test.csv' if m=='final' else INNER/'val_users.csv'
    print(m, 'users:',len(pd.read_csv(path)))

Original train: /kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv | item_meta: True
VAL manifest: f0dfad2f4431a876 | truth: 137041
INNER manifest: b45fa59b9ee68d8a | truth: 109904
val users: 36043
final users: 36043
inner users: 36043


## 2. 원본 JSONL과 관측 이력의 매칭

In [4]:
# 원본의 물리적 행 순서를 유지한 채 사용자·상품별 첫 위치를 기록합니다.
import gzip
from contextlib import contextmanager

@contextmanager
def open_reviews():
    if ORDERING_JSONL:
        path = Path(ORDERING_JSONL)
        opener = gzip.open if path.suffix == '.gz' else open
        with opener(path, 'rb') as stream:
            yield stream
        return
    paths = sorted(p for p in INPUT.rglob('train_reviews.jsonl*')
                   if p.name in {'train_reviews.jsonl', 'train_reviews.jsonl.gz'})
    if paths:
        if len(paths) != 1:
            raise ValueError('원본 JSONL이 여러 개입니다. ORDERING_JSONL에 경로를 지정하세요.')
        with (gzip.open if paths[0].suffix == '.gz' else open)(paths[0], 'rb') as stream:
            yield stream
        return
    matches = []
    for path in sorted(INPUT.rglob('*.zip')):
        if not zipfile.is_zipfile(path):
            continue
        with zipfile.ZipFile(path) as archive:
            matches.extend((path, member) for member in archive.namelist()
                           if Path(member).name == 'train_reviews.jsonl')
    if len(matches) != 1:
        raise ValueError('교수님 원본 train_reviews.jsonl을 Add Input하고 필요하면 ORDERING_JSONL을 지정하세요.')
    path, member = matches[0]
    with zipfile.ZipFile(path) as archive, archive.open(member) as stream:
        yield stream

def load_id_map(path, output_column):
    if path is None:
        return None
    mapping = pd.read_csv(path, dtype={'raw_id': str, output_column: 'int64'})
    assert mapping.raw_id.is_unique and mapping[output_column].is_unique
    return mapping.set_index('raw_id')[output_column].to_dict()

def numeric_id(value, mapping):
    if mapping is not None:
        return mapping.get(str(value), -1)
    if isinstance(value, bool) or not str(value).isdigit():
        raise ValueError('JSONL의 ID가 원본 문자열입니다. ORDERING_USER_MAP / ORDERING_ITEM_MAP에 공식 매핑 CSV를 지정하세요.')
    return int(value)

user_mapping = load_id_map(ORDERING_USER_MAP, 'user_id')
item_mapping = load_id_map(ORDERING_ITEM_MAP, 'item_id')
pair_positions = {}
raw_digest = hashlib.sha256()
with open_reviews() as stream:
    for row_number, line in enumerate(stream):
        raw_digest.update(line)
        if not line.strip():
            continue
        row = json.loads(line)
        raw_user = next((row[key] for key in ('user_id', 'reviewerID', 'reviewer_id') if key in row), None)
        raw_item = next((row[key] for key in ('item_id', 'asin', 'parent_asin') if key in row), None)
        if raw_user is None or raw_item is None:
            raise ValueError(f'JSONL ID 필드를 찾을 수 없습니다. 첫 레코드의 키: {sorted(row)}')
        user = numeric_id(raw_user, user_mapping)
        item = numeric_id(raw_item, item_mapping)
        if user >= 0 and 0 <= item < 19547:
            pair_positions.setdefault(user * 19547 + item, row_number)

position_paths = {}
mapped_digest = hashlib.sha256()
for mode, fit_path in [('inner', INNER/'val_train.csv'), ('val', VAL/'val_train.csv'), ('final', TRAIN)]:
    fit = pd.read_csv(fit_path, usecols=['user_id', 'item_id']).drop_duplicates()
    keys = fit.user_id.to_numpy(np.int64) * 19547 + fit.item_id.to_numpy(np.int64)
    positions = np.fromiter((pair_positions.get(key, -1) for key in keys), dtype=np.int64, count=len(keys))
    missing = positions < 0
    print(mode, 'matched review pairs:', int((~missing).sum()), '/', len(keys))
    if missing.any():
        raise ValueError('학습 사용자·상품 쌍이 JSONL과 매칭되지 않습니다. 원본 파일과 공식 ID 매핑을 확인하세요.')
    result = fit[['user_id']].assign(row_number=positions).groupby('user_id', as_index=False).row_number.min()
    path = WORK/f'ordering_positions_{mode}.csv'
    result.to_csv(path, index=False)
    mapped_digest.update(result.to_numpy(np.int64).tobytes())
    position_paths[mode] = path
del pair_positions
ORDERING_SIGNATURE = [raw_digest.hexdigest(), mapped_digest.hexdigest()]


inner matched review pairs: 368997 / 368997
val matched review pairs: 478901 / 478901
final matched review pairs: 615942 / 615942


## 2. B176 후보와 기준 모델

기존 8개 모델의 출력과 `TEAM_RERANK_EXTRA`를 복원합니다. 원본 로그의 8,510건과 다시 계산한 점수를 모두 기록합니다. 재생성 점수가 다르면 같은 세션 기준 모델과도 비교합니다.

In [5]:

BASE = ['A047_PROXY','A_TEXT','content_fuse_1_0.5_1_k10_pop7','D_VBPR_ENSEMBLE',
        'D_FREEDOM_ENSEMBLE','ease_l500','rp3b_a0.4_b0_k200','D_BLEND6']
ANCHOR = 'TEAM_RERANK_EXTRA'
MODES = ('inner','val','final')
assert HAS_META, '교수 제공 item_meta.jsonl을 Add Input으로 추가하세요.'
assert len(pd.read_csv(VAL/'val_users.csv')) == 36043
assert len(pd.read_csv(VAL/'val_truth.csv')) == 137041

def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

signature = dict(split='f0dfad2f4431a876', inner='b45fa59b9ee68d8a',
                 meta=sha256(ROOT/'data/item_meta.jsonl'), test=sha256(TEST),
                 feature_mode=FEATURE_MODE, version='B187-195-v1', ordering=ORDERING_SIGNATURE, seeds=SEEDS_RANK, threads=THREADS)
signature_path = OUT/'b187_signature.json'
if signature_path.exists():
    assert json.loads(signature_path.read_text()) == signature, '구성이 달라졌습니다. 새 작업 세션을 사용하세요.'
signature_path.write_text(json.dumps(signature, indent=2))

def restore_one(stream, dest):
    temp = dest.with_name(dest.name + '.restore')
    with temp.open('wb') as target:
        shutil.copyfileobj(stream, target)
    temp.replace(dest)

base_files = {f'{name}_{mode}_top100.csv.gz'
              for name in [*BASE, ANCHOR] for mode in MODES}
if REUSE_B176 and FEATURE_MODE != 'proxy':
    cache_root = Path(CACHE_INPUT) if CACHE_INPUT else INPUT
    assert cache_root.is_dir(), cache_root
    for path in sorted(cache_root.rglob('*_top100.csv.gz')):
        if path.name in base_files and not (OUT/path.name).exists():
            with path.open('rb') as stream:
                restore_one(stream, OUT/path.name)
    for path in sorted(cache_root.rglob('TEAM_ALL_TOP100.zip')):
        with zipfile.ZipFile(path) as archive:
            for member in archive.namelist():
                name = Path(member).name
                if name in base_files and not (OUT/name).exists():
                    with archive.open(member) as stream:
                        restore_one(stream, OUT/name)
    fc = ROOT/'data/cache'
    fc.mkdir(exist_ok=True)
    for path in sorted(cache_root.rglob('*.npy')):
        if path.name.startswith(('sbert_all-MiniLM-L6-v2_', 'clip_image_clip-vit-large-patch14', 'clip_text_clip-vit-large-patch14')):
            if not (fc/path.name).exists():
                shutil.copy2(path, fc/path.name)

def verify(name, mode):
    P.verify_top100(name, mode)
    frame = pd.read_csv(OUT/f'{name}_{mode}_top100.csv.gz', usecols=['item_id','score'])
    assert frame.item_id.between(0, 19546).all() and np.isfinite(frame.score).all(), (name, mode)

for name in [*BASE, ANCHOR]:
    for mode in MODES:
        if P.file_ok(name, mode):
            verify(name, mode)

cached_real = ROOT/'data/cache/clip_image_clip-vit-large-patch14.npy'
cache = ROOT/'data/cache'
real_cache_ready = all((cache/f).exists() for f in [
    'clip_image_clip-vit-large-patch14.npy', 'clip_text_clip-vit-large-patch14.npy',
    'sbert_all-MiniLM-L6-v2_tit-sto-cat_256.npy'])
if not real_cache_ready and FEATURE_MODE != 'proxy':
    assert torch.cuda.is_available(), '기존 임베딩 캐시를 추가하거나 GPU를 활성화하세요.'
    if any(importlib.util.find_spec(pkg) is None for pkg in ['sentence_transformers','transformers']):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers', 'transformers'], check=True)


In [6]:
feature_file = SRC / "features.py"
feature_source = feature_file.read_text(encoding="utf-8")

original = """        from common import DATA
        folder=DATA/'images'"""

patched = """        from common import DATA, CACHE
        cache_path = CACHE / f"clip_image_{model_name.split('/')[-1]}.npy"
        if _BACKEND != 'proxy' and cache_path.is_file():
            cached = _original_clip(
                meta, kind=kind, model_name=model_name, batch_size=batch_size
            )
            _record_feature('clip_' + kind, 'real_clip')
            return cached
        folder=DATA/'images'"""

if patched in feature_source:
    print("CLIP 캐시 패치가 이미 적용되어 있습니다.")
elif feature_source.count(original) == 1:
    updated = feature_source.replace(original, patched, 1)
    compile(updated, str(feature_file), "exec")
    feature_file.write_text(updated, encoding="utf-8")
    print("CLIP 캐시 패치 적용 완료")
else:
    raise RuntimeError(f"예상한 CLIP 이미지 분기를 찾을 수 없습니다. {feature_file}")

CLIP 캐시 패치 적용 완료


In [7]:

TEMPORAL = 'B187_TemporalItemKNN'
CONFIGS = [
    ('B188_BinaryTop75', BASE, 75, 'binary'),
    ('B189_BinaryTemporal50', [*BASE, TEMPORAL], 50, 'binary'),
    ('B190_BinaryTemporal75', [*BASE, TEMPORAL], 75, 'binary'),
    ('B191_LambdaRankTemporal50', [*BASE, TEMPORAL], 50, 'lambdarank'),
]
(SRC/'temporal.py').write_text(r'''
"""Time-decayed item co-occurrence from one fit partition."""
import argparse
import numpy as np
import scipy.sparse as sp
import torch
import common as C

DAY_MS = 86_400_000


def recommend(mode, days, window, neighbor_limit, batch):
    fit, users = C.load_fit(mode)
    fit = fit.sort_values(["user_id", "timestamp", "item_id"])
    uid = fit.user_id.to_numpy()
    item = fit.item_id.to_numpy()
    ts = fit.timestamp.to_numpy(dtype=np.float64)
    size = (C.N_ITEMS, C.N_ITEMS)
    graph = sp.csr_matrix(size, dtype=np.float32)
    for lag in range(1, min(window + 1, len(fit))):
        delta = ts[lag:] - ts[:-lag]
        good = (uid[lag:] == uid[:-lag]) & (item[lag:] != item[:-lag]) & (delta <= 4 * days * DAY_MS)
        left, right = item[:-lag][good], item[lag:][good]
        weight = np.exp(-delta[good] / (days * DAY_MS)).astype(np.float32) / np.sqrt(np.float32(lag))
        edge = sp.csr_matrix((weight, (left, right)), shape=size)
        # Random holdout can hide older items, so keep both directions.
        graph = graph + edge + edge.T
    pop = C.popularity(fit)
    scale = np.sqrt(pop + 10)
    rows, cols, values = [], [], []
    for i in range(C.N_ITEMS):
        start, end = graph.indptr[i:i + 2]
        dest = graph.indices[start:end]
        score = graph.data[start:end] / (scale[i] * scale[dest])
        keep = np.lexsort((dest, -score))[:neighbor_limit]
        rows.extend([i] * len(keep))
        cols.extend(dest[keep])
        values.extend(score[keep])
    neighbors = sp.csr_matrix((np.asarray(values, dtype=np.float32), (rows, cols)), shape=size)
    seen = C.interaction_matrix(fit)
    # All visible events vote, with recent events upweighted and older events still retained.
    age = (fit.groupby("user_id")["timestamp"].transform("max").to_numpy(dtype=np.float64) - ts) / DAY_MS
    w = (0.35 + 0.65 * np.exp(-age / 180)).astype(np.float32)
    history = sp.csr_matrix((w, (uid, item)), shape=seen.shape)

    def score(indices):
        return torch.from_numpy((history[indices] @ neighbors).toarray()).to(C.DEVICE)

    items, scores = C.recommend_scores_fn(score, seen, users, pop, topn=100, batch=batch)
    return users, items, scores


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", required=True, choices=C.MODES)
    parser.add_argument("--name", required=True)
    parser.add_argument("--days", type=int, default=180)
    parser.add_argument("--window", type=int, default=8)
    parser.add_argument("--neighbors", type=int, default=200)
    parser.add_argument("--batch", type=int, default=256)
    args = parser.parse_args()
    C.save_outputs(args.name, args.mode, *recommend(args.mode, args.days, args.window, args.neighbors, args.batch))
''')
(SRC/'run187.py').write_text(r'''
import argparse
import gc
import json
import os
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

import common as C
from rerank import Static, build, keys, load_truth_inner


def fit_or_load(modeled, name, sources, depth, objective, seeds, threads):
    paths = [C.RESULTS / f"{name}_s{s}_lgb.txt" for s in seeds]
    if all(p.is_file() for p in paths):
        return [lgb.Booster(model_file=str(path)) for path in paths]

    rows = build("inner", sources, depth, modeled, extra=True)
    rows["label"] = np.isin(keys(rows.user_id.to_numpy(), rows.item_id.to_numpy()), load_truth_inner()).astype(np.int8)
    usable = rows.groupby("user_id", sort=False).label.transform("max").to_numpy() > 0
    rows = rows.loc[usable].reset_index(drop=True)
    users = rows.user_id.unique()
    features = rows.drop(columns=["user_id", "item_id", "label"])
    labels = rows.label.to_numpy()
    row_users = rows.user_id.to_numpy()
    del rows
    models = []
    for seed, path in zip(seeds, paths):
        if path.is_file():
            models.append(lgb.Booster(model_file=str(path)))
            continue
        valid_users = np.random.default_rng(seed).choice(users, max(1, len(users) // 10), replace=False)
        holdout = np.isin(row_users, valid_users)

        def dataset(mask, reference=None):
            selected = row_users[mask]
            group = np.diff(np.r_[0, np.flatnonzero(selected[1:] != selected[:-1]) + 1, len(selected)])
            return lgb.Dataset(features.loc[mask], labels[mask], group=group, reference=reference)

        train = dataset(~holdout)
        valid = dataset(holdout, train)
        params = dict(objective=objective, metric="ndcg", eval_at=[20], learning_rate=0.02,
                      num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8,
                      bagging_fraction=0.8, bagging_freq=1, lambdarank_truncation_level=40,
                      seed=seed, num_threads=threads, force_col_wise=True, verbosity=-1)
        model = lgb.train(params, train, 1200, valid_sets=[valid],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])
        tmp = path.with_suffix(".part")
        model.save_model(str(tmp))
        os.replace(tmp, path)
        models.append(model)
        del train, valid
        gc.collect()
    return models


def save_ranked(rows, pred, anchor, name, mode, users):
    ranked = rows[["user_id", "item_id"]].assign(score=pred)
    ranked.sort_values(["user_id", "score", "item_id"], ascending=[True, False, True], inplace=True)
    ranked = ranked.groupby("user_id", sort=False).head(100)
    counts = ranked.groupby("user_id").size().reindex(users, fill_value=0)
    short = counts.index[counts < 100]
    if len(short):
        fallback = pd.read_csv(C.RESULTS / f"{anchor}_{mode}_top100.csv.gz")
        fallback = fallback[fallback.user_id.isin(short)].sort_values(["user_id", "rank"])
        fallback = fallback.assign(score=-fallback["rank"].astype(np.float32))
        ranked = pd.concat([ranked, fallback[["user_id", "item_id", "score"]]], ignore_index=True)
        ranked = ranked.drop_duplicates(["user_id", "item_id"]).groupby("user_id", sort=False).head(100)
    ranked = ranked.set_index("user_id").loc[users]
    assert len(ranked) == len(users) * 100
    C.save_outputs(name, mode, users, ranked.item_id.to_numpy().reshape(-1, 100),
                   ranked.score.to_numpy(np.float32).reshape(-1, 100))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", required=True)
    ap.add_argument("--sources", required=True, nargs="+")
    ap.add_argument("--anchor", required=True)
    ap.add_argument("--depth", type=int, required=True)
    ap.add_argument("--objective", choices=["binary", "lambdarank"], required=True)
    ap.add_argument("--seeds", nargs="+", type=int, required=True)
    ap.add_argument("--threads", type=int, required=True)
    args = ap.parse_args()
    static = Static()
    models = fit_or_load(static, args.name, args.sources, args.depth, args.objective, args.seeds, args.threads)
    for mode in ["inner", "val", "final"]:
        done = C.RESULTS / f"{args.name}_{mode}.done"
        output = C.RESULTS / f"{args.name}_{mode}_top100.csv.gz"
        if done.exists() and output.is_file():
            continue
        rows = build(mode, args.sources, args.depth, static, extra=True)
        users = pd.read_csv(C.MODES[mode]["users"]).user_id.to_numpy()
        pred = np.zeros(len(rows), dtype=np.float64)
        for model in models:
            pred += model.predict(rows[model.feature_name()], num_threads=args.threads) / len(models)
        save_ranked(rows, pred, args.anchor, args.name, mode, users)
        done.touch()
        del rows, pred
        gc.collect()


if __name__ == "__main__":
    main()
''')

ORDERING_MODELS = {
    'B192_Ordering5': 5,
    'B193_Ordering20': 20,
    'B194_Ordering50': 50,
}
ORDERING_BLEND = 'B195_BinaryOrdering50'
(SRC/'ordering.py').write_text('"""Candidate retrieval from nearby users in the original JSONL order."""\nimport argparse\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nimport common as C\n\n\ndef recommend(mode, positions, window):\n    fit, users = C.load_fit(mode)\n    uid = fit.user_id.to_numpy(np.int64)\n    item = fit.item_id.to_numpy(np.int64)\n    count_users = max(uid.max(), users.max()) + 1\n    first = positions.reindex(np.arange(count_users)).fillna(-1).to_numpy(np.int64)\n    observed = np.unique(uid)\n    if (first[observed] < 0).any():\n        raise ValueError("JSONL 사용자 ID와 train.csv의 매핑을 확인하세요.")\n    ordered = observed[np.argsort(first[observed], kind="stable")]\n    location = np.full(count_users, -1, dtype=np.int64)\n    location[ordered] = np.arange(len(ordered))\n    seen = sp.csr_matrix((np.ones(len(uid), dtype=np.float32), (uid, item)),\n                         shape=(count_users, 19547))\n    seen.data[:] = 1\n    popularity = np.asarray(seen.sum(axis=0)).ravel()\n    penalty = 1 / np.sqrt(1 + popularity)\n    fallback = popularity / max(1, popularity.max())\n    offsets = np.r_[np.arange(-window, 0), np.arange(1, window + 1)]\n    item_ids = np.arange(seen.shape[1])\n    picks = np.empty((len(users), 100), dtype=np.int32)\n    values = np.empty_like(picks, dtype=np.float32)\n    for start in range(0, len(users), 128):\n        batch = users[start:start + 128]\n        centers = location[batch, None]\n        neighbors = centers + offsets\n        valid = (centers >= 0) & (neighbors >= 0) & (neighbors < len(ordered))\n        row, col = np.nonzero(valid)\n        weights = 1 / np.sqrt(np.abs(offsets[col])).astype(np.float32)\n        votes = sp.csr_matrix((weights, (row, ordered[neighbors[row, col]])),\n                              shape=(len(batch), count_users))\n        scores = (votes @ seen).toarray() * penalty\n        scores = np.where(scores > 0, scores, -1 + fallback / 2)\n        seen_rows, seen_items = seen[batch].nonzero()\n        scores[seen_rows, seen_items] = -np.inf\n        for index, score in enumerate(scores):\n            chosen = np.lexsort((item_ids, -score))[:100]\n            picks[start + index] = chosen\n            values[start + index] = score[chosen]\n    assert np.isfinite(values).all()\n    return users, picks, values\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", choices=C.MODES, required=True)\n    parser.add_argument("--name", required=True)\n    parser.add_argument("--positions", required=True)\n    parser.add_argument("--window", type=int, required=True)\n    args = parser.parse_args()\n    positions = pd.read_csv(args.positions).set_index("user_id").row_number\n    C.save_outputs(args.name, args.mode, *recommend(args.mode, positions, args.window))\n')


2753

In [8]:

if RESUME_CHECKPOINT:
    allowed = [TEMPORAL, ANCHOR, *ORDERING_MODELS, ORDERING_BLEND, *(name for name, *_ in CONFIGS)]
    with zipfile.ZipFile(RESUME_CHECKPOINT) as archive:
        assert json.loads(archive.read('b187_signature.json')) == signature
        for member in archive.namelist():
            name = Path(member).name
            if name.startswith(tuple(f'{m}_' for m in allowed)) and name.endswith(('.csv.gz', '.txt', '.done')):
                if not (OUT/name).exists():
                    with archive.open(member) as stream:
                        restore_one(stream, OUT/name)

def rerank(name, sources, depth, objective, seeds=SEEDS_RANK):
    if P.modes_ready(name) and all((OUT/f'{name}_{m}.done').exists() for m in MODES):
        return
    P.run('run187.py',['--name',name,'--sources',*sources,'--anchor',('D_BLEND6' if name==ANCHOR else ANCHOR),
                       '--depth',depth,'--objective',objective,'--seeds',*seeds,'--threads',THREADS], skip=False)



In [9]:

N_ITEMS = 19547
def pairs(frame):
    return frame.user_id.to_numpy(np.int64) * N_ITEMS + frame.item_id.to_numpy(np.int64)

def candidate_audit(mode, configs):
    truth = pd.read_csv(P.truth_path(mode))
    tk = np.unique(pairs(truth))
    depths = {}
    for _, sources, depth in configs:
        for name in sources:
            depths.setdefault(name, set()).add(depth)
    chunks = {}
    for name, cuts in depths.items():
        top = pd.read_csv(OUT/f'{name}_{mode}_top100.csv.gz',
                          usecols=['user_id','item_id','rank'], dtype=np.int32)
        keys = pairs(top)
        ranks = top['rank'].to_numpy()
        chunks[name] = {depth: keys[ranks <= depth] for depth in cuts}
    data, baseline_hits = [], None
    for name, sources, depth in configs:
        union = np.unique(np.concatenate([chunks[s][depth] for s in sources]))
        hits = np.intersect1d(union, tk, assume_unique=True)
        if baseline_hits is None:
            baseline_hits = hits
        capped = np.minimum(np.bincount(hits // N_ITEMS),20).sum()
        data.append(dict(mode=mode, model=name, candidates=len(union),
                         candidates_per_user=len(union)/36043, candidate_hits=len(hits),
                         candidate_recall=len(hits)/len(truth), **{'oracle_recall@20': capped/len(truth)},
                         added_hits=len(np.setdiff1d(hits,baseline_hits,assume_unique=True)),
                         lost_hits=len(np.setdiff1d(baseline_hits,hits,assume_unique=True))))
    return data


def write_audit(configs):
    table = pd.DataFrame([row for mode in ('inner','val')
                          for row in candidate_audit(mode, configs)])
    table.to_csv(WORK/'candidate_audit.csv', index=False)
    print(table.to_string(index=False))
    return table


In [10]:

def checkpoint():
    target = WORK/'B187_195_checkpoint.zip'
    temp = target.with_suffix('.part')
    prefixes = tuple(f'{n}_' for n in [ANCHOR,TEMPORAL,*ORDERING_MODELS,ORDERING_BLEND,*(c[0] for c in CONFIGS)])
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        z.write(signature_path,signature_path.name)
        for path in sorted(OUT.iterdir()):
            if path.name.startswith(prefixes) and path.name.endswith(('_lgb.txt','_top100.csv.gz','.done')):
                z.write(path,path.name)
    temp.replace(target)

results_path = WORK/'ablation_results.csv'
def record():
    names = [ANCHOR,TEMPORAL,*ORDERING_MODELS,ORDERING_BLEND,*(c[0] for c in CONFIGS)]
    rows = []
    for name in names:
        if not P.modes_ready(name):
            continue
        result = P.evaluate(name)
        config = next((c for c in [*CONFIGS, *ordering_config] if c[0]==name), None)
        if name in ORDERING_MODELS:
            parameters = f'window=±{ORDERING_MODELS[name]} users, inverse square root distance and popularity discount'
        elif config:
            parameters = f'{config[3]}, per-source top{config[2]}, lr=.02, leaves=31, min_leaf=500, seeds={SEEDS_RANK}'
            if name == ORDERING_BLEND:
                parameters += f', ordering_source={config[1][-1]}'
        elif name == ANCHOR:
            parameters = 'binary top50, extra cosine, seeds 0/1/2'
        else:
            parameters = 'window=8, decay_days=180, neighbor_limit=200' 
        result.update(experiment_id=('B176' if name==ANCHOR else name.split('_')[0]),
                      parameters=parameters,
                      delta_vs_historical_b176=result['recall@20']-8510/137041,
                      test=np.nan)
        rows.append(result)
    frame = pd.DataFrame(rows)
    frame.to_csv(results_path,index=False)
    return frame


## 3. Ordering 단독 비교

앞뒤 5·20·50명의 관측 상품을 거리와 인기도로 가중합니다.

In [11]:

for name, window in ORDERING_MODELS.items():
    for mode in MODES:
        done = OUT/f'{name}_{mode}.done'
        if not done.exists() or not P.file_ok(name, mode):
            P.run('ordering.py', ['--mode', mode, '--name', name,
                                 '--positions', position_paths[mode], '--window', window],
                  expected=name, mode=mode, skip=False)
            verify(name, mode)
            done.touch()
    print(P.evaluate(name))
    checkpoint()


RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/ordering.py --mode inner --name B192_Ordering5 --positions /kaggle/working/ordering_positions_inner.csv --window 5
DONE (s): 87
Verified Top100 B192_Ordering5 inner rows 3604300
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/ordering.py --mode val --name B192_Ordering5 --positions /kaggle/working/ordering_positions_val.csv --window 5
DONE (s): 88
Verified Top100 B192_Ordering5 val rows 3604300
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/ordering.py --mode final --name B192_Ordering5 --positions /kaggle/working/ordering_positions_final.csv --window 5
DONE (s): 90
Verified Top100 B192_Ordering5 final rows 3604300
{'model': 'B192_Ordering5', 'mode': 'val', 'recall@20': 0.012638553425617151, 'hits': 1732, 'truth_count': 137041, 'users': 36043}
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/ordering.py --mode inner --name B193_Ordering20 --positions /kaggle/working/ordering_positions_in

In [12]:
# 멀티모달 가용성 검사. 가능하면 사전학습 MiniLM/CLIP 사용; 불가능하면 proxy로 자동 전환.
import requests
images=ROOT/'data/images'
images.mkdir(parents=True,exist_ok=True)
if HAS_META and DOWNLOAD_IMAGES and FEATURE_MODE!='proxy' and not cached_real.exists():
    try:
        url=None
        with (ROOT/'data/item_meta.jsonl').open(encoding='utf-8') as f:
            for line in f:
                obj=json.loads(line)
                for im in (obj.get('images') or []):
                    url=next((im[k] for k in ['large','hi_res','thumb'] if im.get(k)),None)
                    if url:break
                if url:break
        if url:
            r=requests.get(url,timeout=6,headers={'User-Agent':'Mozilla/5.0'},stream=True)
            r.close()
            if r.status_code==200:
                print('이미지 서버 접속 가능 → 이미지 다운로드 (중단 시 기존 이미지 재사용)')
                P.run('download_images.py',['--workers','12'],skip=False,optional=True)
            else:print('이미지 서버 응답',r.status_code,'→ 이미지 proxy 사용 가능')
        else:print('item_meta에 이미지 URL 없음. 이미지 proxy 사용 가능')
    except Exception as ex:
        print('이미지 다운로드 생략:',repr(ex))
print('현재 이미지 파일:',len(list(images.glob('*.jpg'))))
# 서로 다른 실험 설정의 캐시를 실수로 혼합하지 않도록 방지
signature={'preset':PRESET,'feature_mode':FEATURE_MODE,'download_images':DOWNLOAD_IMAGES}
sigfile=ROOT/'pipeline_signature.json'
if sigfile.exists():
    old=json.loads(sigfile.read_text(encoding='utf8'))
    if old!=signature:
        raise RuntimeError(f'기존 모델 파일은 다른 설정으로 생성됐습니다: {old}. '
                           '새 Kaggle 세션을 사용하거나 /kaggle/working/team_ensemble 폴더를 정리하세요.')
else:sigfile.write_text(json.dumps(signature,indent=2),encoding='utf8')
if not HAS_META:print('⚠ item_meta.jsonl 부재. A text/meta, VBPR/FREEDOM 및 원래 D reranker는 재현할 수 없습니다.')

이미지 서버 접속 가능 → 이미지 다운로드 (중단 시 기존 이미지 재사용)
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/download_images.py --workers 12
to download: 19543, no url: 4
1000/19543 done, failed 4
2000/19543 done, failed 4
3000/19543 done, failed 4
4000/19543 done, failed 4
5000/19543 done, failed 4
6000/19543 done, failed 4
7000/19543 done, failed 4
8000/19543 done, failed 4
9000/19543 done, failed 4
10000/19543 done, failed 4
11000/19543 done, failed 4
12000/19543 done, failed 4
13000/19543 done, failed 4
14000/19543 done, failed 4
15000/19543 done, failed 4
16000/19543 done, failed 4
17000/19543 done, failed 4
18000/19543 done, failed 4
19000/19543 done, failed 4
finished. failed/missing: 4
DONE (s): 36
현재 이미지 파일: 19543


In [13]:
if not all(P.modes_ready(n) for n in BASE):
    A_MODELS=['A_BM25','A_TEXT','A_META','A047_PROXY']
    for mode in ['inner','val','final']:
        if not all(P.file_ok(n,mode) for n in ['A047_PROXY','A_TEXT']):
            P.run('a_proxy.py',['--mode',mode]+(['--no-meta'] if not HAS_META else []),skip=False)
    print('A 모델 완료:',{x:P.modes_ready(x) for x in A_MODELS})
    if P.file_ok('A047_PROXY','val'):print('A047_PROXY',P.evaluate('A047_PROXY'))

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/a_proxy.py --mode inner
A_PROXY mode=inner: users 36043, interactions 368997
 A_PROXY inner 8192/36043
 A_PROXY inner 16384/36043
 A_PROXY inner 24576/36043
 A_PROXY inner 32768/36043
 A_PROXY inner 36043/36043
saved A_BM25 inner
saved A_TEXT inner
saved A_META inner
saved A047_PROXY inner
DONE (s): 209
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/a_proxy.py --mode val
A_PROXY mode=val: users 36043, interactions 478901
 A_PROXY val 8192/36043
 A_PROXY val 16384/36043
 A_PROXY val 24576/36043
 A_PROXY val 32768/36043
 A_PROXY val 36043/36043
saved A_BM25 val
saved A_TEXT val
saved A_META val
saved A047_PROXY val
DONE (s): 209
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/a_proxy.py --mode final
A_PROXY mode=final: users 36043, interactions 615942
 A_PROXY final 8192/36043
 A_PROXY final 16384/36043
 A_PROXY final 24576/36043
 A_PROXY final 32768/36043
 A_PROXY final 36043/36043
saved A_BM25 final

In [14]:
if not all(P.modes_ready(n) for n in BASE):
    D_CONTENT='content_fuse_1_0.5_1_k10_pop7'
    if HAS_META:
        for mode in ['inner','val','final']:
            P.run('run_content.py',['--features','fuse_1_0.5_1','--k','10','--pop-beta','7','--mode',mode,'--no-log'],
                  expected=D_CONTENT,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
    print('D_CONTENT 3 modes:',P.modes_ready(D_CONTENT))

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/run_content.py --features fuse_1_0.5_1 --k 10 --pop-beta 7 --mode inner --no-log


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 1654.86it/s]


features fuse_1_0.5_1: shape (19547, 1920), 357.7s
wrote /kaggle/working/team_ensemble/D/results/content_fuse_1_0.5_1_k10_pop7_inner.csv
DONE (s): 408
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/run_content.py --features fuse_1_0.5_1 --k 10 --pop-beta 7 --mode val --no-log
features fuse_1_0.5_1: shape (19547, 1920), 1.4s
[content_fuse_1_0.5_1_k10_pop7] recall@20=0.04208 hits=5767 groups={'<=13': 0.04537, '14-18': 0.04356, '>=19': 0.03785}
DONE (s): 53
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/run_content.py --features fuse_1_0.5_1 --k 10 --pop-beta 7 --mode final --no-log
features fuse_1_0.5_1: shape (19547, 1920), 1.5s
wrote /kaggle/working/team_ensemble/D/results/content_fuse_1_0.5_1_k10_pop7_final.csv
DONE (s): 49
D_CONTENT 3 modes: True


In [15]:
if not all(P.modes_ready(n) for n in BASE):
    SEEDS=[0,1,2] if PRESET=='MAX' else [0]
    VBPR_E=200 if PRESET=='MAX' else 100
    VBPR_NAMES=[]
    if HAS_META and not P.modes_ready('D_VBPR_ENSEMBLE'):
        for seed in SEEDS:
            name=f'vbpr_fuse_d64v64_lr0.001_reg0.005_e{VBPR_E}_s{seed}'
            for mode in ['inner','val','final']:
                P.run('vbpr.py',['--features','fuse','--d','64','--d-vis','64','--lr','0.001','--reg','0.005',
                               '--epochs',str(VBPR_E),'--eval-every',str(VBPR_E),'--seed',str(seed),'--mode',mode,'--no-log'],
                      expected=name,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
            if P.modes_ready(name):VBPR_NAMES.append(name)
    print('VBPR seeds ready:',VBPR_NAMES)

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/vbpr.py --features fuse --d 64 --d-vis 64 --lr 0.001 --reg 0.005 --epochs 200 --eval-every 200 --seed 0 --mode inner --no-log
epoch 1 loss 0.6705 (1.3s)
epoch 2 loss 0.5912 (0.8s)
epoch 3 loss 0.5372 (0.8s)
epoch 4 loss 0.4929 (0.8s)
epoch 5 loss 0.4505 (0.8s)
epoch 6 loss 0.4108 (0.8s)
epoch 7 loss 0.3749 (0.8s)
epoch 8 loss 0.3429 (0.8s)
epoch 9 loss 0.3146 (0.8s)
epoch 10 loss 0.2917 (0.8s)
epoch 11 loss 0.2712 (0.8s)
epoch 12 loss 0.2530 (0.8s)
epoch 13 loss 0.2388 (0.8s)
epoch 14 loss 0.2252 (0.8s)
epoch 15 loss 0.2137 (0.8s)
epoch 16 loss 0.2042 (0.8s)
epoch 17 loss 0.1958 (0.8s)
epoch 18 loss 0.1878 (0.8s)
epoch 19 loss 0.1800 (0.8s)
epoch 20 loss 0.1753 (0.8s)
epoch 21 loss 0.1698 (0.8s)
epoch 22 loss 0.1640 (0.8s)
epoch 23 loss 0.1602 (0.8s)
epoch 24 loss 0.1560 (0.8s)
epoch 25 loss 0.1525 (0.8s)
epoch 26 loss 0.1495 (0.8s)
epoch 27 loss 0.1466 (0.8s)
epoch 28 loss 0.1440 (0.8s)
epoch 29 loss 0.1410 (0.8s)
epoch 30 l

In [16]:
if not all(P.modes_ready(n) for n in BASE):
    FREEDOM_E=100 if PRESET=='MAX' else 75
    FREEDOM_NAMES=[]
    if HAS_META and not P.modes_ready('D_FREEDOM_ENSEMBLE'):
        for seed in SEEDS:
            name=f'freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e{FREEDOM_E}_s{seed}'
            for mode in ['inner','val','final']:
                P.run('freedom.py',['--d','64','--n-layers','2','--n-mm-layers','1','--knn-k','10',
                             '--w-img','0.1','--dropout','0.8','--reg-weight','0.0001','--lr','0.001',
                             '--epochs',str(FREEDOM_E),'--eval-every',str(FREEDOM_E),'--seed',str(seed),
                             '--mode',mode,'--no-log'],
                      expected=name,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
            if P.modes_ready(name):FREEDOM_NAMES.append(name)
    print('FREEDOM seeds ready:',FREEDOM_NAMES)

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 0 --mode inner --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6902 (4.5s)
epoch 10 loss 0.3625 (4.3s)
epoch 20 loss 0.2496 (4.2s)
epoch 30 loss 0.1748 (4.2s)
epoch 40 loss 0.1261 (4.2s)
epoch 50 loss 0.0954 (4.2s)
epoch 60 loss 0.0759 (4.2s)
epoch 70 loss 0.0635 (4.2s)
epoch 80 loss 0.0534 (4.2s)
epoch 90 loss 0.0468 (4.2s)
epoch 100 loss 0.0420 (4.2s)
wrote /kaggle/working/team_ensemble/D/results/freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s0_inner.csv
DONE (s): 461
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 0 --mode val --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6806 (5.8s)
epoch 10 loss 0.3625 (5.7s)
epoch 20 loss 0.2606 (5.7s)
epoch 30 loss 0.1845 (5.7s)
epoch 40 loss 0.1359 (5.7s)
epoch 50 loss 0.1065 (5.7s)
epoch 60 loss 0.0863 (5.7s)
epoch 70 loss 0.0721 (5.7s)
epoch 80 loss 0.0630 (5.7s)
epoch 90 loss 0.0555 (5.7s)
epoch 100 loss 0.0498 (5.7s)
[freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s0] recall@20=0.04701 hits=6442 groups={'<=13': 0.04851, '14-18': 0.04711, '>=19': 0.04576}
DONE (s): 612
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 0 --mode final --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


... (9 outputs left out)

/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6816 (5.8s)
epoch 10 loss 0.3586 (5.7s)
epoch 20 loss 0.2577 (5.7s)
epoch 30 loss 0.1847 (5.7s)
epoch 40 loss 0.1363 (5.7s)
epoch 50 loss 0.1058 (5.7s)
epoch 60 loss 0.0857 (5.7s)
epoch 70 loss 0.0716 (5.7s)
epoch 80 loss 0.0619 (5.7s)
epoch 90 loss 0.0545 (5.7s)
epoch 100 loss 0.0493 (5.7s)
[freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s2] recall@20=0.04625 hits=6338 groups={'<=13': 0.04765, '14-18': 0.04631, '>=19': 0.04514}
DONE (s): 612
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/freedom.py --d 64 --n-layers 2 --n-mm-layers 1 --knn-k 10 --w-img 0.1 --dropout 0.8 --reg-weight 0.0001 --lr 0.001 --epochs 100 --eval-every 100 --seed 2 --mode final --no-log


/kaggle/working/team_ensemble/D/src/freedom.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()


epoch 1 loss 0.6607 (7.8s)
epoch 10 loss 0.3538 (7.7s)
epoch 20 loss 0.2626 (7.7s)
epoch 30 loss 0.1908 (7.7s)
epoch 40 loss 0.1438 (7.7s)
epoch 50 loss 0.1137 (7.7s)
epoch 60 loss 0.0945 (7.7s)
epoch 70 loss 0.0798 (7.7s)
epoch 80 loss 0.0713 (7.7s)
epoch 90 loss 0.0634 (7.7s)
epoch 100 loss 0.0578 (7.7s)
wrote /kaggle/working/team_ensemble/D/results/freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s2_final.csv
DONE (s): 810
FREEDOM seeds ready: ['freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s0', 'freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s1', 'freedom_d64_L2_k10_wi0.1_do0.8_rw0.0001_wd0_lr0.001_e100_s2']


In [17]:
if not all(P.modes_ready(n) for n in BASE):
    D_CF={
    'ease_l500': ['ease','--lam','500'],
    'rp3b_a0.4_b0_k200':['rp3b','--alpha','0.4','--beta','0','--k','200'],
    }
    for name,args in D_CF.items():
        for mode in ['inner','val','final']:
            P.run('cf.py',[*args,'--mode',mode,'--no-log'],expected=name,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
    print('CF ready:',{n:P.modes_ready(n) for n in D_CF})

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ease --lam 500 --mode inner --no-log
wrote /kaggle/working/team_ensemble/D/results/ease_l500_inner.csv
DONE (s): 65
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ease --lam 500 --mode val --no-log
[ease_l500] recall@20=0.04345 hits=5955 groups={'<=13': 0.04201, '14-18': 0.04263, '>=19': 0.04552}
DONE (s): 71
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py ease --lam 500 --mode final --no-log
wrote /kaggle/working/team_ensemble/D/results/ease_l500_final.csv
DONE (s): 66
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py rp3b --alpha 0.4 --beta 0 --k 200 --mode inner --no-log
wrote /kaggle/working/team_ensemble/D/results/rp3b_a0.4_b0_k200_inner.csv
DONE (s): 58
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/cf.py rp3b --alpha 0.4 --beta 0 --k 200 --mode val --no-log
[rp3b_a0.4_b0_k200] recall@20=0.04232 hits=5799 groups={'<=13': 0.0427, '14-18': 0.04179

In [18]:
if not all(P.modes_ready(n) for n in BASE):
    def seed_ensemble(names,alias):
        if not names:return None
        if len(names)==1:
            for mode in ['inner','val','final']:
                P.rename_output(names[0],alias,mode)
        else:
            for mode in ['inner','val','final']:
                P.run('blend.py',['--inputs',*names,'--name',alias,'--mode',mode,'--no-log'],
                      expected=alias,mode=mode,optional=ALLOW_OPTIONAL_FAILURE)
        return alias if P.modes_ready(alias) else None
    
    V_ALIAS='D_VBPR_ENSEMBLE' if P.modes_ready('D_VBPR_ENSEMBLE') else seed_ensemble(VBPR_NAMES,'D_VBPR_ENSEMBLE')
    F_ALIAS='D_FREEDOM_ENSEMBLE' if P.modes_ready('D_FREEDOM_ENSEMBLE') else seed_ensemble(FREEDOM_NAMES,'D_FREEDOM_ENSEMBLE')
    print('seed ensemble:',V_ALIAS,F_ALIAS)
    
    BASE_PARTS=[];BASE_WEIGHTS=[]
    for name,w in [(D_CONTENT,1.0),(V_ALIAS,0.7),(F_ALIAS,1.3),('ease_l500',1.0),
                   ('rp3b_a0.4_b0_k200',0.0),('ials_d128_a10_r100_it15_s0',0.0)]:
        if name and P.modes_ready(name) and w>0:
            BASE_PARTS.append(name);BASE_WEIGHTS.append(w)
    if len(BASE_PARTS)<2:
        BASE_PARTS=['A047_PROXY','A_BM25'];BASE_WEIGHTS=[1.,0.3]
    for mode in ['inner','val','final']:
        P.run('blend.py',['--inputs',*BASE_PARTS,'--weights',*map(str,BASE_WEIGHTS),
                          '--rrf-c','60','--name','D_BLEND6','--mode',mode,'--no-log'],
              expected='D_BLEND6',mode=mode,optional=False)
    print('D_BLEND6',P.evaluate('D_BLEND6'))

RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s0 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s1 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s2 --name D_VBPR_ENSEMBLE --mode inner --no-log
wrote /kaggle/working/team_ensemble/D/results/D_VBPR_ENSEMBLE_inner.csv
DONE (s): 49
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s0 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s1 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s2 --name D_VBPR_ENSEMBLE --mode val --no-log
[D_VBPR_ENSEMBLE] recall@20=0.03796 hits=5202 groups={'<=13': 0.03717, '14-18': 0.03557, '>=19': 0.04142}
DONE (s): 53
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/blend.py --inputs vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s0 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s1 vbpr_fuse_d64v64_lr0.001_reg0.005_e200_s2 --name D_VBPR_ENSEMBLE --mode final --no-log
wrote /kaggle/working/team_ensemble/D/results/D_

In [19]:
assert all(P.modes_ready(n) for n in BASE), 'B176 원천 후보 일부 누락'
for name in BASE:
    for mode in MODES:
        verify(name,mode)


Verified Top100 A047_PROXY inner rows 3604300
Verified Top100 A047_PROXY val rows 3604300
Verified Top100 A047_PROXY final rows 3604300
Verified Top100 A_TEXT inner rows 3604300
Verified Top100 A_TEXT val rows 3604300
Verified Top100 A_TEXT final rows 3604300
Verified Top100 content_fuse_1_0.5_1_k10_pop7 inner rows 3604300
Verified Top100 content_fuse_1_0.5_1_k10_pop7 val rows 3604300
Verified Top100 content_fuse_1_0.5_1_k10_pop7 final rows 3604300
Verified Top100 D_VBPR_ENSEMBLE inner rows 3604300
Verified Top100 D_VBPR_ENSEMBLE val rows 3604300
Verified Top100 D_VBPR_ENSEMBLE final rows 3604300
Verified Top100 D_FREEDOM_ENSEMBLE inner rows 3604300
Verified Top100 D_FREEDOM_ENSEMBLE val rows 3604300
Verified Top100 D_FREEDOM_ENSEMBLE final rows 3604300
Verified Top100 ease_l500 inner rows 3604300
Verified Top100 ease_l500 val rows 3604300
Verified Top100 ease_l500 final rows 3604300
Verified Top100 rp3b_a0.4_b0_k200 inner rows 3604300
Verified Top100 rp3b_a0.4_b0_k200 val rows 3604300

## 4. B176 후보와 Ordering 결합

기존 8개 모델의 Top-50에 Ordering Top-50을 더해 같은 Binary 설정으로 학습합니다.

In [20]:

ordering_audit_configs = [('B176', BASE, 50)]
ordering_audit_configs.extend((name, [name], 100) for name in ORDERING_MODELS)
ordering_audit_configs.extend((f'B176_plus_{name}', [*BASE, name], 50) for name in ORDERING_MODELS)
initial_audit = write_audit(ordering_audit_configs)
# 결합에 사용할 윈도우는 Inner에서만 선택합니다. 공통 Validation은 성능 보고에 사용합니다.
selection = initial_audit[(initial_audit['mode'] == 'inner') &
                          initial_audit.model.str.startswith('B176_plus_')]
ordered = selection.sort_values('oracle_recall@20', ascending=False, kind='stable')
ordering_source = ordered.iloc[0].model.removeprefix('B176_plus_')
ordering_config = [(ORDERING_BLEND, [*BASE, ordering_source], 50, 'binary')]
print('Selected on inner:', ordering_source)
if not P.modes_ready(ANCHOR):
    # Same candidate features and split. CPU backend and padding can differ on reconstruction.
    rerank(ANCHOR, BASE, 50, 'binary')


rerank(*ordering_config[0])
for mode in MODES:
    verify(ORDERING_BLEND, mode)
print(record().sort_values('recall@20', ascending=False).to_string(index=False))
checkpoint()


 mode                     model  candidates  candidates_per_user  candidate_hits  candidate_recall  oracle_recall@20  added_hits  lost_hits
inner                      B176     7802798           216.485809           18587          0.169120          0.169120           0          0
inner            B192_Ordering5     3604300           100.000000            3866          0.035176          0.035176        2081      16802
inner           B193_Ordering20     3604300           100.000000            2187          0.019899          0.019899        1718      18118
inner           B194_Ordering50     3604300           100.000000            2221          0.020209          0.020209        1736      18102
inner  B176_plus_B192_Ordering5     9579516           265.780207           19914          0.181194          0.181194        1327          0
inner B176_plus_B193_Ordering20     9580767           265.814916           19942          0.181449          0.181449        1355          0
inner B176_plus_B194

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 7,802,798 (216.5/user) in 30s
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.160263
[200]	valid_0's ndcg@20: 0.163236
[300]	valid_0's ndcg@20: 0.162322
Early stopping, best iteration is:
[214]	valid_0's ndcg@20: 0.16341
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.170913
[200]	valid_0's ndcg@20: 0.173361
[300]	valid_0's ndcg@20: 0.17492
Early stopping, best iteration is:
[261]	valid_0's ndcg@20: 0.175729
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.168597
[200]	valid_0's ndcg@20: 0.171532
[300]	valid_0's ndcg@20: 0.171994
[400]	valid_0's ndcg@20: 0.173219
[500]	valid_0's ndcg@20: 0.172314
Early stopping, best iteration is:
[460]	valid_0's ndcg@20: 0.173449
[inner] candidates 7,802,798 (216.5/user) in 29s
[val] candidates 7,514,315 (208.5/user) in 28s
[final] candidates 7,288,337 (202.2/user) in 28s
DONE (s): 915
RUN: /usr/bin/python3 -u /k

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 9,579,127 (265.8/user) in 36s
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.204943
Early stopping, best iteration is:
[46]	valid_0's ndcg@20: 0.20607
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.19638
[200]	valid_0's ndcg@20: 0.199522
[300]	valid_0's ndcg@20: 0.199544
[400]	valid_0's ndcg@20: 0.200231
[500]	valid_0's ndcg@20: 0.200999
[600]	valid_0's ndcg@20: 0.201241
[700]	valid_0's ndcg@20: 0.200678
Early stopping, best iteration is:
[634]	valid_0's ndcg@20: 0.201624
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.198786
[200]	valid_0's ndcg@20: 0.200013
[300]	valid_0's ndcg@20: 0.199777
[400]	valid_0's ndcg@20: 0.200483
Early stopping, best iteration is:
[354]	valid_0's ndcg@20: 0.201163
[inner] candidates 9,579,127 (265.8/user) in 34s
[val] candidates 9,292,268 (257.8/user) in 34s
[final] candidates 9,067,434 (251.6/user) in 34s
DONE (s)

## 3. 시간 가중 ItemKNN

8개 이내의 이력 위치 차이를 가지는 상품 쌍을 시간 간격과 위치 차이로 감쇠합니다. 공통 split이 무작위 홀드아웃이므로 과거와 미래 양쪽으로 연결합니다. 상품 인기도로 보정한 뒤 품목별 200개 이웃을 남기고, 사용자의 이력 전체에서 최근 상품에 더 큰 가중치를 부여합니다. 동점은 인기 신호로 처리합니다.

In [21]:

if RUN_TEMPORAL:
    for mode in MODES:
        done = OUT/f'{TEMPORAL}_{mode}.done'
        if not done.exists() or not P.file_ok(TEMPORAL, mode):
            P.run('temporal.py',['--mode',mode,'--name',TEMPORAL,'--days',180,'--window',8,
                                '--neighbors',200,'--batch',256],expected=TEMPORAL,mode=mode,skip=False)
            verify(TEMPORAL,mode)
            done.touch()

    write_audit([*ordering_audit_configs, (TEMPORAL, [TEMPORAL], 100),
                 *((name, models, depth) for name, models, depth, _ in CONFIGS
                   if name != 'B191_LambdaRankTemporal50')])
    record()
    checkpoint()
    for name, sources, depth, objective in CONFIGS:
        rerank(name, sources, depth, objective)
        for mode in MODES:
            verify(name, mode)
        print(record().sort_values('recall@20', ascending=False).to_string(index=False))
        checkpoint()


RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/temporal.py --mode inner --name B187_TemporalItemKNN --days 180 --window 8 --neighbors 200 --batch 256
DONE (s): 40
Verified Top100 B187_TemporalItemKNN inner rows 3604300
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/temporal.py --mode val --name B187_TemporalItemKNN --days 180 --window 8 --neighbors 200 --batch 256
DONE (s): 41
Verified Top100 B187_TemporalItemKNN val rows 3604300
RUN: /usr/bin/python3 -u /kaggle/working/team_ensemble/D/src/temporal.py --mode final --name B187_TemporalItemKNN --days 180 --window 8 --neighbors 200 --batch 256
DONE (s): 42
Verified Top100 B187_TemporalItemKNN final rows 3604300
 mode                     model  candidates  candidates_per_user  candidate_hits  candidate_recall  oracle_recall@20  added_hits  lost_hits
inner                      B176     7802798           216.485809           18587          0.169120          0.169120           0          0
inner            B192_Ord

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 11,690,451 (324.3/user) in 37s
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.130088
[200]	valid_0's ndcg@20: 0.132649
[300]	valid_0's ndcg@20: 0.133406
[400]	valid_0's ndcg@20: 0.134494
Early stopping, best iteration is:
[398]	valid_0's ndcg@20: 0.134577
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.134507
[200]	valid_0's ndcg@20: 0.137671
[300]	valid_0's ndcg@20: 0.137926
[400]	valid_0's ndcg@20: 0.138705
Early stopping, best iteration is:
[326]	valid_0's ndcg@20: 0.139012
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.132245
[200]	valid_0's ndcg@20: 0.133596
Early stopping, best iteration is:
[177]	valid_0's ndcg@20: 0.13447
[inner] candidates 11,690,451 (324.3/user) in 36s
[val] candidates 11,247,006 (312.0/user) in 35s
[final] candidates 10,847,488 (301.0/user) in 35s
DONE (s): 1317
Verified Top100 B188_BinaryTop75 inner rows 3604300
Ver

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 9,301,501 (258.1/user) in 35s
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.165825
[200]	valid_0's ndcg@20: 0.168544
[300]	valid_0's ndcg@20: 0.169239
[400]	valid_0's ndcg@20: 0.169905
[500]	valid_0's ndcg@20: 0.17085
[600]	valid_0's ndcg@20: 0.171615
[700]	valid_0's ndcg@20: 0.171483
Early stopping, best iteration is:
[604]	valid_0's ndcg@20: 0.171687
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.151794
[200]	valid_0's ndcg@20: 0.153532
[300]	valid_0's ndcg@20: 0.154134
[400]	valid_0's ndcg@20: 0.15445
[500]	valid_0's ndcg@20: 0.15478
Early stopping, best iteration is:
[441]	valid_0's ndcg@20: 0.155296
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.147542
[200]	valid_0's ndcg@20: 0.148958
[300]	valid_0's ndcg@20: 0.151633
[400]	valid_0's ndcg@20: 0.151508
Early stopping, best iteration is:
[348]	valid_0's ndcg@20: 0.152602
[inner] candidate

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 13,824,190 (383.5/user) in 43s
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.126745
[200]	valid_0's ndcg@20: 0.127249
[300]	valid_0's ndcg@20: 0.128046
Early stopping, best iteration is:
[222]	valid_0's ndcg@20: 0.128327
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.130674
[200]	valid_0's ndcg@20: 0.133483
[300]	valid_0's ndcg@20: 0.134181
[400]	valid_0's ndcg@20: 0.134178
Early stopping, best iteration is:
[329]	valid_0's ndcg@20: 0.135188
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.130985
[200]	valid_0's ndcg@20: 0.132443
Early stopping, best iteration is:
[195]	valid_0's ndcg@20: 0.13272
[inner] candidates 13,824,190 (383.5/user) in 42s
[val] candidates 13,447,484 (373.1/user) in 42s
[final] candidates 13,018,303 (361.2/user) in 41s
DONE (s): 1411
Verified Top100 B190_BinaryTemporal75 inner rows 3604300
Verified Top100 B190_BinaryTempo

/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dtype=torch.int64),
/kaggle/working/team_ensemble/D/src/rerank.py:152: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  Xt = torch.sparse_csr_tensor(torch.as_tensor(X.indptr, dtype=torch.int64), torch.as_tensor(X.indices, dty

[inner] candidates 9,301,501 (258.1/user) in 35s
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.164957
Early stopping, best iteration is:
[74]	valid_0's ndcg@20: 0.16773
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.156206
[200]	valid_0's ndcg@20: 0.157335
Early stopping, best iteration is:
[138]	valid_0's ndcg@20: 0.158233
Training until validation scores don't improve for 100 rounds
[100]	valid_0's ndcg@20: 0.148589
Early stopping, best iteration is:
[46]	valid_0's ndcg@20: 0.150456
[inner] candidates 9,301,501 (258.1/user) in 34s
[val] candidates 9,035,533 (250.7/user) in 34s
[final] candidates 8,771,251 (243.4/user) in 33s
DONE (s): 729
Verified Top100 B191_LambdaRankTemporal50 inner rows 3604300
Verified Top100 B191_LambdaRankTemporal50 val rows 3604300
Verified Top100 B191_LambdaRankTemporal50 final rows 3604300
                    model mode  recall@20  hits  truth_count  users experiment_id    

## 6. 최종 선택, 제출 형식 검증, 공유 ZIP

최고 공통 Validation 점수를 선택합니다. 동점이면 B176을 우선합니다. 결괏값은 사용자 단위 paired bootstrap 및 활동량 그룹별로도 확인합니다. 동일 split에서 여러 설정을 고르므로 bootstrap 구간은 선택 편향이나 Kaggle 점수를 보장하지 않습니다.

In [22]:

from itertools import chain

table = record()
winner = table.sort_values('recall@20',ascending=False,kind='stable').iloc[0]['model']
truth = pd.read_csv(P.truth_path('val'))
denom = np.bincount(truth.user_id,minlength=36043)
tk = np.unique(pairs(truth))
def user_hits(name):
    top = pd.read_csv(OUT/f'{name}_val_top100.csv.gz',usecols=['user_id','item_id','rank'])
    top = top[top['rank'] <= 20]
    return np.bincount(top.user_id[np.isin(pairs(top),tk)],minlength=36043)

h0 = user_hits(ANCHOR)
activity = np.bincount(pd.read_csv(VAL/'val_train.csv',usecols=['user_id']).user_id,minlength=36043)
rng = np.random.default_rng(187)
boot_idx = [rng.integers(0,len(denom),size=len(denom),dtype=np.int32) for _ in range(300)]
diagnostics = []
for name in table.model:
    hits = user_hits(name)
    delta = hits-h0
    ci = np.quantile([delta[idx].sum()/denom[idx].sum() for idx in boot_idx],[0.025,0.975])
    row = dict(model=name, delta_vs_rerun_b176=delta.sum()/denom.sum(), paired_ci_low=ci[0], paired_ci_high=ci[1])
    for group, mask in [('le13',activity<=13),('14to18',(activity>=14)&(activity<=18)),('ge19',activity>=19)]:
        row[f'recall_{group}'] = hits[mask].sum()/max(1,denom[mask].sum())
    diagnostics.append(row)
pd.DataFrame(diagnostics).to_csv(WORK/'validation_stability.csv',index=False)
verify(winner,'final')
submission = P.write_top20_submission(winner,'final',WORK/'submission.csv')
sub = pd.read_csv(submission)
test_users = pd.read_csv(TEST).user_id
assert sub.columns.tolist() == ['user_id','item_id'] and sub.user_id.equals(test_users)
items = sub.item_id.str.split().tolist()
assert all(len(r)==len(set(r))==20 for r in items)
flat = np.fromiter(map(int,chain.from_iterable(items)),dtype=np.int64)
assert ((flat>=0)&(flat<N_ITEMS)).all()
assert not np.isin(np.repeat(test_users.to_numpy(np.int64),20)*N_ITEMS+flat,
                   pairs(pd.read_csv(TRAIN,usecols=['user_id','item_id']))).any()
table.sort_values('recall@20',ascending=False).to_csv(WORK/'validation_scores.csv',index=False)
notion = table[table.experiment_id != 'B176'][['experiment_id','model','parameters','recall@20','test']]
notion = notion.rename(columns={'model':'model_or_combination','recall@20':'val_recall@20'})
notion.to_csv(WORK/'notion_results.csv',index=False,float_format='%.6f',na_rep='')
manifest_rows = []
zip_path = WORK/'B187_B195_results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for name in table.model:
        for mode in MODES:
            path = OUT/f'{name}_{mode}_top100.csv.gz'
            share_name = ('B176_' if name==ANCHOR else '') + path.name
            assert path.is_file()
            z.write(path,'top100/'+share_name)
            manifest_rows.append(dict(model=name,mode=mode,original_filename=path.name,share_filename=share_name,
                                      inner_in_sample=(mode=='inner' and name not in {TEMPORAL, *ORDERING_MODELS})))
    pd.DataFrame(manifest_rows).to_csv(WORK/'top100_manifest.csv',index=False)
    for filename in ['submission.csv','ablation_results.csv','validation_scores.csv','candidate_audit.csv',
                     'validation_stability.csv','notion_results.csv','top100_manifest.csv']:
        path = WORK/filename
        assert path.is_file() and path.stat().st_size>0
        z.write(path,filename)
with zipfile.ZipFile(zip_path) as z:
    assert len(z.namelist()) == 3*len(table)+7
print('Selected:',winner,'| Submission:',submission,'| Outputs:',zip_path)


Verified Top100 B195_BinaryOrdering50 final rows 3604300
Submission: /kaggle/working/submission.csv | users 36,043
Selected: B195_BinaryOrdering50 | Submission: /kaggle/working/submission.csv | Outputs: /kaggle/working/B187_B195_results.zip
